In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

In [7]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
# ── Configuration ──────────────────────────────────────────
COL_X   = "nb_windows"
COL_Y   = "value"
EXCLUDE = ["timestamp", "chunk", "notes", "file_id"]

FOLDER_NAME = "csv_meteo_and_soil"    
RESULTS_CSV_PATH = os.path.join("/content/gdrive/MyDrive/NIFA_Download/outputs", FOLDER_NAME, "results.csv")

In [ ]:

# ── Construction des widgets de filtre ──────────────────────
def build_filters():
    df_temp = pd.read_csv(RESULTS_CSV_PATH)
    params = [
        c for c in df_temp.columns
        if c not in [COL_X, COL_Y] + EXCLUDE
        and df_temp[c].nunique() > 1
        and not df_temp[c].isna().all()
    ]
    filtres = {}
    for col in params:
        valeurs = ["Tous"]
        if df_temp[col].isna().any():
            valeurs.append("NaN")
        valeurs += sorted(df_temp[col].dropna().unique().tolist(), key=str)
        filtres[col] = widgets.SelectMultiple(
            options=valeurs,
            value=["Tous"],
            description=col,
            layout=widgets.Layout(width="500px", height="120px")
        )
    return filtres, params
filtres, PARAMS = build_filters()
btn = widgets.Button(description="Mettre à jour", button_style="primary")
output = widgets.Output()
def tracer(b):
    with output:
        output.clear_output(wait=True)
        # ← Recharge le CSV à chaque clic
        df = pd.read_csv(RESULTS_CSV_PATH)
        masque = pd.Series([True] * len(df), index=df.index)
        for col, w in filtres.items():
            sel = list(w.value)
            if "Tous" not in sel:
                col_mask = pd.Series([False] * len(df), index=df.index)
                if "NaN" in sel:
                    col_mask |= df[col].isna()
                    autres = [v for v in sel if v != "NaN"]
                    if autres:
                        col_mask |= df[col].isin(autres)
                else:
                    col_mask = df[col].isin(sel)
                masque &= col_mask
        sous_df = df[masque].copy()
        cols_var = [c for c in PARAMS if c in sous_df.columns and sous_df[c].nunique() > 1]
        sous_df["_combo"] = sous_df[cols_var].apply(
            lambda r: " | ".join(f"{k}={v}" for k, v in zip(cols_var, r)), axis=1
        )
        # ── Figure avec espace réservé pour la légende ──
        fig, ax = plt.subplots(figsize=(20, 12))
        fig.subplots_adjust(right=0.70)   # 30% à droite pour la légende
        for label, grp in sous_df.groupby("_combo"):
            g = grp.sort_values(COL_X)
            ax.plot(g[COL_X], g[COL_Y], label=label,
                    marker='o', markersize=5, linewidth=1.5, alpha=0.75)
        ax.set_xlabel(COL_X)
        ax.set_ylabel(COL_Y)
        ax.set_title(f"{COL_Y} vs {COL_X} — {len(sous_df)} points filtrés")
        ax.legend(
            title="Paramètres",
            bbox_to_anchor=(1.02, 1), loc="upper left",
            frameon=True, fancybox=False
        )
        plt.show()
btn.on_click(tracer)
ui = widgets.HBox(list(filtres.values()))
display(ui, btn, output)
tracer(None)